# Create ML-ready dataset from FH-EARLY export

> Goal: build a reusable pipeline that converts exported eCRF tables into one ML dataframe
> 
> NB: This notebook uses mock data only to test the structure.. final modelling decisions will be made after receiving the real dataset
> 
> We will use: 
> - export `.csv`, which contain the actual data row
> - `master_catalog`, that we created in `tutorial_catalog.ipynb`that tell us how columns should be interpreted: 
>   - identify numerical, categorical, checkbox, date and texts
>   - chose analysis ready columns
>   - keep track of feature meaning

In [25]:
import importlib
from pathlib import Path
import utils_database

importlib.reload(utils_database)

from utils_database import *

export_folder = Path("export-example")

master_catalog = pd.read_pickle("master_catalog.pkl")

### Load catalog

In [9]:
master_catalog.shape
master_catalog.head()

,table,column,sas_label,sas_informat,Référence,Id,Saisie,Type,Format,Intitulé,Type_english,code_labels
0,INC,STUDY_ID,Identification étude,$8.,STUDY_ID,NaN,NaN,NaN,NaN,NaN,unknown,{}
1,INC,COUNTRY_ID,Identifiant pays,$4.,COUNTRY_ID,NaN,NaN,NaN,NaN,NaN,unknown,{}
2,INC,EXTRACTION_DATE,Date extraction,$19.,EXTRACTION_DATE,NaN,NaN,NaN,NaN,NaN,unknown,{}
3,INC,SITE_ID,Centre,$4.,SITE_ID,NaN,NaN,NaN,NaN,NaN,unknown,{}
4,INC,SUBJECT_ID,Identifiant unique,$4.,SUBJECT_ID,NaN,NaN,NaN,NaN,NaN,unknown,{}


In [10]:
master_catalog[["Type_english"]].value_counts(dropna=False)
# If you want to analize the unknown types, you can do it like this:
# master_catalog[master_catalog["Type_english"]== "unknown"]
# Probably these are not unknown because parsing failed, but because they are administrative identifiers, technical export cokumns, module and do not belong to the clinical dictionary. 

Type_english
numeric         124
categorical      87
date             64
checkbox         54
unknown          37
text             15
Name: count, dtype: int64

### Load the `.csv file`

In [27]:
inc = read_export_csv(export_folder / "INC_ANSI.csv")
ml = read_export_csv(export_folder / "ML_ANSI.csv")
tab_bs = read_export_csv(export_folder / "TAB_BS_ANSI.csv")
tab_fh = read_export_csv(export_folder / "TAB_FH_ANSI.csv")

export_tables = {"INC": inc, "ML": ml, "TAB_BS": tab_bs, "TAB_FH": tab_fh}

### Create table-specific analysis datasets
> Before merging everything into one dataset, we create one analysis-ready dataset per exported table, so we can: 
> - inspect each module indipendentely
> - test preprocessing rule 
> - then use identifiers to merge
> 
> To create the first version of the dataset we use `create_table_dataset` and we keep only variable which `Type_english` (information that can be found in the `catalog`) is not unknown AND `_ID` variables to then merge the dataset


In [28]:
master_catalog["table"].value_counts()

table
ML        272
INC        73
TAB_BS     20
TAB_FH     16
Name: count, dtype: int64

In [29]:
inc_dataset = create_table_dataset(inc, "INC", master_catalog)
ml_dataset = create_table_dataset(ml, "ML", master_catalog)
tab_bs_dataset = create_table_dataset(tab_bs, "TAB_BS", master_catalog)
tab_fh_dataset = create_table_dataset(tab_fh, "TAB_FH", master_catalog)

In [30]:
print("INC:", inc_dataset.shape)
print("ML:", ml_dataset.shape)
print("TAB_BS:", tab_bs_dataset.shape)
print("TAB_FH:", tab_fh_dataset.shape)

INC: (5, 68)
ML: (5, 268)
TAB_BS: (5, 14)
TAB_FH: (5, 10)


### Inspect the generated datasets
> The generated datasets contain only columns considered potentially useful for analysis:
> - numeric
> - categorical
> - checkbox
> - date
> - text variables
> 
> Technical/system columns excluded from the metadata catalog are not included

In [32]:
ml_dataset.head()

,STUDY_ID,COUNTRY_ID,SITE_ID,SUBJECT_ID,SUBJECT_REF,CD1TXT,CD2LS,CD3NUM,CD3NUM_V,CD4NUM,...,TTT4LS_C1,TTT4LS_C2,TTT4LS_C3,TTT4LS_C4,TTT4LS_C5,TTT4LS_C6,TTT4LS_C7,VS1YN,VS1ALS,VS1BTXT
0,FH-EARLY,fr,1,c1/1,ZZOI-0001,SIOIQBDS,2,66,66.0,150.823,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN
1,FH-EARLY,fr,2,c1/2,QGYD-0002,BGRXGOEYMZRG,2,113,113.0,111.952,...,1.0,1.0,NaN,NaN,NaN,NaN,NaN,1,3.0,NaN
2,FH-EARLY,fr,3,c1/3,EJI-0003,NXQBDZBPXB,2,106,106.0,133.242,...,1.0,1.0,1.0,1.0,1.0,NaN,NaN,1,6.0,NaN
3,FH-EARLY,fr,4,c1/4,CH-0004,XWRXMDZTWJ,1,74,74.0,196.870,...,1.0,1.0,1.0,NaN,NaN,NaN,NaN,0,NaN,NaN
4,FH-EARLY,fr,5,c1/5,UHVT-0005,RO,2,123,123.0,NaN,...,1.0,1.0,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN


## Handle variable representations

> Different exported variable types require different preprocessing strategies before creating ml datasets
>
> Examples:
> - date variables may need conversion into timestamps or derived temporal features
> - categorical variables may require label or one-hot encoding
> - checkbox variables are already expanded into binary columns
> - numeric variables may require selecting the `_V` analysis-ready columns

In [33]:
ml_numeric_cols = get_numeric_cols(master_catalog, "ML") # keep only numeric column with _v when available
ml_categorical_cols = get_categorical_cols(master_catalog, "ML")
ml_checkbox_cols = get_checkbox_cols(master_catalog, "ML")
inc_date_cols = get_date_cols(master_catalog, "INC") # keep the main date columns, not the single year, month, day columns
ml_text_cols = get_text_cols(master_catalog, "ML")

In [34]:
ml_features = summarize_table_features(master_catalog, "ML")

for k, v in ml_features.items():
    print(k, len(v))

numeric 59
categorical 62
checkbox 54
date 6
text 6


In [35]:
ml_ready = create_ml_table_dataset(ml, master_catalog, "ML")
ml_ready.head()

,STUDY_ID,COUNTRY_ID,SITE_ID,SUBJECT_ID,SUBJECT_REF,CD3NUM_V,CD4NUM_V,CD5NUM_V,CD6NUM_V,CD7NUM_V,...,BS8ADT,BS30ADT,BS36ADT,BS37ADT,BS38ADT,CD1TXT,BS1A1TXT,BS34A1TXT,BS35A1TXT,VS1BTXT
0,FH-EARLY,fr,1,c1/1,ZZOI-0001,66.0,150.823,29.0,112.0,117.0,...,09/07/2020,NaN,NaN,NaN,06/03/1970,SIOIQBDS,NaN,NaN,NaN,NaN
1,FH-EARLY,fr,2,c1/2,QGYD-0002,113.0,111.952,90.2,202.0,NaN,...,04/03/2015,NaN,NaN,NaN,NaN,BGRXGOEYMZRG,NaN,NaN,"inclusion traitement clinique , inclusion visite",NaN
2,FH-EARLY,fr,3,c1/3,EJI-0003,106.0,133.242,59.7,187.0,84.0,...,NaN,NaN,NaN,NaN,14/08/2006,NXQBDZBPXB,NaN,NaN,NaN,NaN
3,FH-EARLY,fr,4,c1/4,CH-0004,74.0,196.870,19.1,70.0,77.0,...,NaN,03/03/1996,03/06/1987,01/05/2013,22/08/1971,XWRXMDZTWJ,inclusion médicament,NaN,. visite ? inclusion clinique essai essai suivi,NaN
4,FH-EARLY,fr,5,c1/5,UHVT-0005,123.0,NaN,3.4,132.0,41.0,...,27/10/1999,NaN,26/04/1992,NaN,04/09/1972,RO,NaN,NaN,NaN,NaN


## Inspect missing values

In [40]:
missing_summary = (ml_ready.isna().mean().sort_values(ascending=False).to_frame("missing_fraction"))
missing_summary

,missing_fraction
VS1BTXT,1.0
MH5E1NUM_V,1.0
TTT4LS_C7,1.0
TTT4LS_C6,1.0
BS34CNUM_V,1.0
...,...
FH3YN,0.0
MH5YN,0.0
MH3YN,0.0
MH2YN,0.0
